# Transfer Learning para Clasificacion de Biomas de la Amazonia Peruana

**Universidad Nacional del Altiplano - Ciencia de Datos**

Este notebook implementa un clasificador de biomas amazonicos usando Transfer Learning con ResNet18 pre-entrenada en ImageNet.

**Dataset**: BiomePeruvianAmazon2023 (14 clases de ecosistemas amazonicos)

In [ ]:
# 1. Instalar kagglehub y descargar dataset
!pip install -q kagglehub

import kagglehub

# Descargar dataset
DATASET_PATH = kagglehub.dataset_download("earthshot/biomeperuvianamazon2023")
print(f"Dataset descargado en: {DATASET_PATH}")

In [ ]:
# 2. Explorar estructura del dataset
import os

print(f"Contenido de {DATASET_PATH}:")
for item in os.listdir(DATASET_PATH):
    path = os.path.join(DATASET_PATH, item)
    if os.path.isdir(path):
        n = len(os.listdir(path))
        print(f"  [DIR] {item}/ ({n} elementos)")
    else:
        print(f"  [FILE] {item}")

In [ ]:
# 3. Encontrar carpeta con clases de imagenes
import os

def find_classes_folder(base):
    for root, dirs, _ in os.walk(base):
        valid = 0
        for d in dirs:
            subdir = os.path.join(root, d)
            if any(f.lower().endswith(('.jpg','.jpeg','.png')) for f in os.listdir(subdir)):
                valid += 1
        if valid >= 2:
            return root
    return base

# Encontrar ruta correcta
DATA_PATH = find_classes_folder(DATASET_PATH)
print(f"\nRuta de datos: {DATA_PATH}")

# Mostrar clases
classes = sorted([d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))])
print(f"\nClases encontradas ({len(classes)}):")
for c in classes:
    n = len([f for f in os.listdir(os.path.join(DATA_PATH, c)) if f.lower().endswith(('.jpg','.png','.jpeg'))])
    print(f"  {c}: {n} imagenes")

In [ ]:
# 4. Imports y configuracion
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import random

# Reproducibilidad
random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 5. Preparar datos con data augmentation
IMG_SIZE, BATCH_SIZE = 224, 32
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Transformaciones con aumento de datos para entrenamiento
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Transformaciones sin aumento para validacion
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Cargar dataset completo para split
full_dataset = ImageFolder(root=DATA_PATH, transform=train_tf)
CLASSES = full_dataset.classes
NUM_CLASSES = len(CLASSES)

# Split 80/20
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Clases: {NUM_CLASSES}")
print(f"Train: {len(train_ds)} imagenes | Val: {len(val_ds)} imagenes")

In [ ]:
# 6. Visualizar distribucion de clases
from collections import Counter

# Contar imagenes por clase
class_counts = {}
for c in CLASSES:
    class_path = os.path.join(DATA_PATH, c)
    class_counts[c] = len([f for f in os.listdir(class_path) if f.lower().endswith(('.jpg','.png','.jpeg'))])

# Grafico de barras
plt.figure(figsize=(14, 6))
bars = plt.bar(range(len(CLASSES)), list(class_counts.values()), color='forestgreen', alpha=0.8)
plt.xticks(range(len(CLASSES)), CLASSES, rotation=45, ha='right')
plt.xlabel('Clase de Bioma')
plt.ylabel('Numero de Imagenes')
plt.title('Distribucion de Imagenes por Clase de Bioma Amazonico')
plt.tight_layout()
plt.savefig('distribucion_clases.png', dpi=150)
plt.show()

print(f"\nTotal de imagenes: {sum(class_counts.values())}")

In [ ]:
# 7. Visualizar muestras del dataset
def denormalize(img):
    """Desnormalizar imagen para visualizacion"""
    img = img.numpy().transpose((1, 2, 0))
    return np.clip(np.array(std) * img + np.array(mean), 0, 1)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
imgs, lbls = next(iter(train_loader))
for i in range(10):
    ax = axes[i // 5, i % 5]
    ax.imshow(denormalize(imgs[i]))
    ax.set_title(CLASSES[lbls[i]][:20], fontsize=9)
    ax.axis('off')
plt.suptitle('Muestras del Dataset BiomePeruvianAmazon2023', fontsize=12)
plt.tight_layout()
plt.savefig('muestras_dataset.png', dpi=150)
plt.show()

In [ ]:
# 8. Configurar modelo ResNet18 con Transfer Learning
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Congelar todas las capas convolucionales
for param in model.parameters():
    param.requires_grad = False

# Reemplazar capa clasificadora
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, NUM_CLASSES)
)

model = model.to(DEVICE)

# Contar parametros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Modelo ResNet18 configurado para {NUM_CLASSES} clases")
print(f"Parametros totales: {total_params:,}")
print(f"Parametros entrenables: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

In [ ]:
# 9. Entrenamiento
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

EPOCHS = 10
train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = 0

print("Iniciando entrenamiento...\n")
for epoch in range(EPOCHS):
    # Fase de entrenamiento
    model.train()
    running_loss, correct, total = 0, 0, 0
    
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * x.size(0)
        correct += (outputs.argmax(1) == y).sum().item()
        total += y.size(0)
    
    train_losses.append(running_loss / total)
    train_accs.append(correct / total)
    
    # Fase de validacion
    model.eval()
    running_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            outputs = model(x)
            loss = criterion(outputs, y)
            running_loss += loss.item() * x.size(0)
            correct += (outputs.argmax(1) == y).sum().item()
            total += y.size(0)
    
    val_losses.append(running_loss / total)
    val_accs.append(correct / total)
    
    # Guardar mejor modelo
    if val_accs[-1] > best_val_acc:
        best_val_acc = val_accs[-1]
        torch.save(model.state_dict(), 'mejor_modelo.pth')
    
    scheduler.step()
    
    print(f"Epoca {epoch+1}/{EPOCHS} | Train Loss: {train_losses[-1]:.4f} | Train Acc: {train_accs[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} | Val Acc: {val_accs[-1]:.4f}")

print(f"\nEntrenamiento completado! Mejor Val Accuracy: {best_val_acc:.4f}")

In [ ]:
# 10. Curvas de aprendizaje
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(range(1, EPOCHS+1), train_losses, 'b-o', label='Entrenamiento', linewidth=2, markersize=6)
ax1.plot(range(1, EPOCHS+1), val_losses, 'r-o', label='Validacion', linewidth=2, markersize=6)
ax1.set_xlabel('Epoca')
ax1.set_ylabel('Perdida (Loss)')
ax1.set_title('Curva de Perdida')
ax1.legend()
ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(range(1, EPOCHS+1), train_accs, 'b-o', label='Entrenamiento', linewidth=2, markersize=6)
ax2.plot(range(1, EPOCHS+1), val_accs, 'r-o', label='Validacion', linewidth=2, markersize=6)
ax2.set_xlabel('Epoca')
ax2.set_ylabel('Precision (Accuracy)')
ax2.set_title('Curva de Precision')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Curvas de Aprendizaje - Transfer Learning ResNet18', fontsize=12)
plt.tight_layout()
plt.savefig('curvas_aprendizaje.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 11. Matriz de confusion y metricas
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        outputs = model(x)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

# Matriz de confusion
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASSES, yticklabels=CLASSES,
            annot_kws={'size': 8})
plt.xlabel('Prediccion', fontsize=12)
plt.ylabel('Valor Real', fontsize=12)
plt.title('Matriz de Confusion - Clasificacion de Biomas Amazonicos', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 12. Reporte de clasificacion detallado
print("=" * 80)
print("REPORTE DE CLASIFICACION - BIOMAS DE LA AMAZONIA PERUANA")
print("=" * 80)
print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))

# Precision por clase
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(all_labels, all_preds, average=None)
recall = recall_score(all_labels, all_preds, average=None)
f1 = f1_score(all_labels, all_preds, average=None)

print("\n" + "=" * 80)
print("METRICAS POR CLASE")
print("=" * 80)
for i, clase in enumerate(CLASSES):
    print(f"{clase:30s} | Precision: {precision[i]:.4f} | Recall: {recall[i]:.4f} | F1: {f1[i]:.4f}")

In [ ]:
# 13. Visualizar predicciones correctas e incorrectas
model.eval()

# Obtener batch de validacion
imgs, lbls = next(iter(val_loader))
with torch.no_grad():
    outputs = model(imgs.to(DEVICE))
    preds = outputs.argmax(1).cpu()

# Encontrar correctas e incorrectas
correct_idx = [i for i in range(len(preds)) if preds[i] == lbls[i]][:5]
incorrect_idx = [i for i in range(len(preds)) if preds[i] != lbls[i]][:5]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Correctas
for i, idx in enumerate(correct_idx):
    if i < 5:
        axes[0, i].imshow(denormalize(imgs[idx]))
        axes[0, i].set_title(f'OK: {CLASSES[preds[idx]][:15]}', fontsize=8, color='green')
        axes[0, i].axis('off')

# Incorrectas
for i, idx in enumerate(incorrect_idx):
    if i < 5:
        axes[1, i].imshow(denormalize(imgs[idx]))
        axes[1, i].set_title(f'Pred: {CLASSES[preds[idx]][:10]}\nReal: {CLASSES[lbls[idx]][:10]}', fontsize=7, color='red')
        axes[1, i].axis('off')

plt.suptitle('Predicciones Correctas (arriba) e Incorrectas (abajo)', fontsize=12)
plt.tight_layout()
plt.savefig('predicciones_ejemplo.png', dpi=150)
plt.show()

In [ ]:
# 14. Guardar modelo final
checkpoint = {
    'model_state_dict': model.state_dict(),
    'classes': CLASSES,
    'num_classes': NUM_CLASSES,
    'train_accs': train_accs,
    'val_accs': val_accs,
    'train_losses': train_losses,
    'val_losses': val_losses
}
torch.save(checkpoint, 'modelo_biomas_amazonia.pth')

print(f"Modelo guardado exitosamente!")
print(f"Precision final de validacion: {val_accs[-1]:.2%}")
print(f"Mejor precision de validacion: {best_val_acc:.2%}")

In [ ]:
# 15. Descargar archivos (solo en Google Colab)
try:
    from google.colab import files
    print("Descargando archivos...")
    files.download('modelo_biomas_amazonia.pth')
    files.download('curvas_aprendizaje.png')
    files.download('matriz_confusion.png')
    files.download('distribucion_clases.png')
    files.download('muestras_dataset.png')
    files.download('predicciones_ejemplo.png')
except ImportError:
    print("No se esta ejecutando en Colab. Los archivos estan guardados localmente.")